<a href="https://colab.research.google.com/github/pardhasaradhi96/Video_func/blob/main/Videofunc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Metadata
input = "" # @param {"type":"string","placeholder":"enter file path"}
!ffmpeg -i "{input}"

In [ ]:
#@title mediainfo
import subprocess
def install_mediainfo():
  try:
    subprocess.run(["mediainfo", "--version"], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  except FileNotFoundError:
    print("mediainfo is not installed. Installing now...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "mediainfo"], check=True)
install_mediainfo()
import re
#=======================  display streams   ====================================
def parse_mediainfo_fields(filepath):
    result = subprocess.check_output(['mediainfo', filepath], text=True)
    lines = result.splitlines()

    streams = []
    current = {}
    stream_type = ""

    for line in lines:
        if not line.strip():
            continue
        if re.match(r'^(General|Video|Audio|Text)', line):
            if current:
                streams.append((stream_type, current))
            stream_type = line.strip()
            current = {}
        elif ':' in line:
            k, v = map(str.strip, line.split(':', 1))
            current[k] = v
    if current:
        streams.append((stream_type, current))
    return streams
# STEP 4: Clean & Display
def clean_size(size):
    # Remove inbuilt parentheses with percentages
    return re.sub(r'\s*\(.*?\)', '', size).strip()

def clean_resolution(w, h):
    return f"{w.replace(' ', '')}x{h.replace(' ', '')}".replace("pixels", "")

def extract_stream_id(stream_title):
    # extract the last number after '#'
    match = re.search(r'#(\d+)', stream_title)
    return match.group(1) if match else ""

def print_formatted_summary(streams):
    for i,(stream_type, data) in enumerate(streams):
        try:
          id_ = int(str(data.get("ID", ""))) - 1
        except ValueError:
          print()
        title = data.get("Title", "")
        size = clean_size(data.get("Stream size", ""))
        duration = data.get("Duration", "")
        bit_depth = data.get("Bit depth", "")
        hdr_tag = data.get("HDR format", "").strip()  # Only use this now
        if stream_type.startswith("Video"):
            res = clean_resolution(data.get("Width", ""), data.get("Height", ""))
            output = f"{id_}: ({data.get('Format', '')}), ({bit_depth}), ({size}), ({res}), ({duration}), ({title})"
            if hdr_tag:  # Only add if HDR format exists
                output += f", ({hdr_tag})"
            print(output)
        elif stream_type.startswith("Audio"):
            lang = data.get("Language", "")
            layout = data.get("Channel layout", "")
            Channel = data.get("Channel(s)", "")
            bitrate = data.get("Bit rate", "")
            fmt = data.get("Format", "")
            profile = data.get("Format profile", "")
            full_fmt = f"{fmt} {profile}".strip()
            stream_name = f"Audio #{extract_stream_id(stream_type)}"
            print(f"{id_}: ({stream_name}) ({lang}), ({size}), ({full_fmt}), ({Channel}), ({bitrate}), ({duration}), ({title}) ,  ({layout})")
        elif stream_type.startswith("Text"):
            lang = data.get("Language", "")
            codec = data.get("Codec ID", "")
            fmt = data.get("Format", "")
            print(f"{id_}: (Subtitle) ({lang}), ({fmt}), ({codec}), ({duration}), ({title})")
def find_file_size(filepath):
    size_bytes = os.path.getsize(filepath)
    for unit in ["B","KiB","MiB","GiB","TiB"]:
        if size_bytes < 1024:
            return f"{size_bytes:.2f} {unit}"
        size_bytes /= 1024
    return f"{size_bytes:.2f} PiB"
filepath="" #@param{"type":"string"}
streams = parse_mediainfo_fields(filepath)
print("**** streams ******")
print("total size = ",find_file_size(filepath))
print_formatted_summary(streams)

In [ ]:
#@title multiple files duration
import os
import subprocess
import json
def get_video_duration_subprocess(video_path):
    try:
        command = [
            "ffprobe",
            "-v", "quiet",
            "-show_format",
            "-print_format", "json",
            video_path
        ]
        result = subprocess.run(command, capture_output=True, text=True, check=True)
        ffprobe_data = json.loads(result.stdout)
        duration = float(ffprobe_data["format"]["duration"])
        return duration
    except subprocess.CalledProcessError as e:
        print(f"ffprobe error: {e.stderr}")
        return None
    except json.JSONDecodeError:
        print("Error decoding JSON from ffprobe output.")
        return None
    except KeyError:
        print("Could not find duration in ffprobe output.")
        return None
process_files=[]
input_files="" #@param{"type":"string","placeholder":"paste folder path or file paths separate by comma"}
if os.path.isdir(input_files):
    for f in os.listdir(input_files):
            process_files.append(os.path.join(input_files, f))
elif "," in input_files:
    for f in input_files.split(","):
        process_files.append(f.strip())
for p in process_files:
  print(f"{get_video_duration_subprocess(p)} = {os.path.basename(p)}")

In [ ]:
# @title Generate Sample
import os
import random
import subprocess
import json
import shutil
from google.colab import files
def install_mkvmerge():
      try:
          # Check if mkvmerge is installed
          subprocess.run(["mkvmerge", "--version"], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
      except FileNotFoundError:
          # If not installed, print a message and install it using apt
          print("mkvmerge is not installed. Installing now...")
          subprocess.run(["sudo", "apt-get", "update", "-qq"], check=True)
          subprocess.run(["sudo", "apt-get", "install", "-y", "mkvtoolnix"], check=True)
      except subprocess.CalledProcessError as e:
          print(f"An error occurred while checking mkvmerge: {e}")
install_mkvmerge()
shutil.rmtree("sample", ignore_errors=True)
if input in locals():
  del input
# List of video files (modify this with your filenames)
input_files="" # @param {"type":"string","placeholder":"paste file paths separated by commas"}
def remove_ipynb_checkpoints(list_of_folders):
  folders=list_of_folders
  for folder in folders:
    ipynb_path = os.path.join(folder, ".ipynb_checkpoints")
    if os.path.isdir(ipynb_path):
        shutil.rmtree(ipynb_path)
if os.path.isdir(input_files):
  remove_ipynb_checkpoints([input_files])
  video_files = [os.path.join(input_files, file) for file in os.listdir(input_files)]
else:
  video_files = [file.strip() for file in input_files.split(",") if file.strip()]
# Set duration of the sample (in seconds)
duration = "" # @param {"type":"string","placeholder":"enter duration in seconds"}
if duration=="":
  duration = 60
else:
  duration = int(duration)
startTime = ""#@param{"type":"string","placeholder":"enter start time (optional)"}
endTime = ""#@param{"type":"string","placeholder":"enter end time (optional)"}
downloadfile=""  #@param {"type":"string","placeholder":"enter 1 download first file , enter 2 for download all files "}
multiple="" #@param {"type":"string","placeholder":"enter number for multiple samples ex: 3 (means 3 samples)"}
oldfilesremove="" #@param {"type":"string","placeholder":"enter 1 to remove contents in output folder"}
if oldfilesremove=="1":
    shutil.rmtree(outputfolder, ignore_errors=True)
# Function to get video duration using ffprobe
def get_video_duration(video):
    cmd = [
        "ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries",
        "format=duration", "-of", "json", video
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    try:
        duration = float(json.loads(result.stdout)["format"]["duration"])
    except (KeyError, ValueError, json.JSONDecodeError):
        duration = 0
    return duration

random_time=""
howlong=""
def randomtimegeneration():
  min_duration = min(get_video_duration(video) for video in video_files)
  # Ensure random start time is within valid range
  min_duration = float(min_duration)  # Convert to float to avoid errors
  random_time = str(random.randint(0, int(min_duration) - duration) if min_duration > duration else 0)
  return random_time
if endTime:
    howlong = ["-to", endTime]  # Split "-to" and the value into separate elements
else:
    howlong = ["-t", str(duration)]
# Create output directory
output_dir = "samples"
os.makedirs(output_dir, exist_ok=True)
downloaded=0
samples_list = []
# Process each video file
times=1
if multiple and multiple.isdigit():
  times=int(multiple)
for t in range(times):
  if startTime:
    random_time=startTime
  else:
      random_time = randomtimegeneration()
  print("start time = ",random_time)
  for video in video_files:
      filename, ext = os.path.splitext(os.path.basename(video))
      output_file = os.path.join(output_dir, f"{random_time}_{filename}_sample{t}{ext}")
      op="1.mkv"
      # FFmpeg command
      cmd = [
          "ffmpeg",
          "-i", video,
          "-ss", random_time,
          *howlong,
          "-map", "0",
          "-c", "copy",
          "-y",
          op
      ]
      subprocess.run(cmd, check=True)
      # Mkvmerge command
      cmd2 = ["mkvmerge", "-o", output_file, op]
      subprocess.run(cmd2, check=True)
      samples_list.append(output_file)
      os.remove(op)
      if downloadfile=="1" and downloaded==0:
        files.download(output_file)
        downloaded=1
      if downloadfile=="2":
        files.download(output_file)
downloaded=0
print("files = ", ",".join(samples_list))
print("Video samples generated successfully!")
startTime = ""
endTime = ""
downloadfile=""
multiple=""

In [ ]:
from google.colab import files
import os
import subprocess
import re
import random
import shutil
#-------------------------------- extra functions ---------------------------
KEYFRAMES_DIR = "keyframes"
def naturalSortKey(path: str):
    convert = lambda c: int(c) if c.isdigit() else c.lower()
    return [convert(chunk) for chunk in re.split(r"(\d+)", path)]
def processTheVideos(videos: str):
    videos_list=[]
    if os.path.isdir(videos): #ex: /videos
        videos_list = sorted(
            [os.path.join(videos, f) for f in os.listdir(videos)
             if os.path.isfile(os.path.join(videos, f))],key=naturalSortKey)
    elif ":" in videos: #ex: /videos:video1.mkv,video2.mkv,/pictures:sample1.mkv
        videos_list = []
        parts = videos.split(',')
        current_dir = ''
        for part in parts:
            if ':' in part:
                current_dir, filename = part.split(':', 1)
                videos_list.append(f"{current_dir}/{filename}")
            elif os.path.isfile(part):
              videos_list.append(part)
            else:
                videos_list.append(f"{current_dir}/{part}")
    else: #ex: : /videos/video1.mkv,/videos/video2.mkv
      videos_list = [f.strip() for f in videos.split(",") if f.strip()]
    return videos_list
def execution(cmd):
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(f"Command failed with return code {result.returncode}")

def get_video_duration(video_path):
    try:
        result = subprocess.run([
            'ffprobe', '-v', 'error', '-select_streams', 'v:0',
            '-show_entries', 'format=duration',
            '-of', 'default=noprint_wrappers=1:nokey=1',
            video_path
        ], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, check=True)
        return float(result.stdout.strip())
    except:
        return float('inf')

def get_file_size(video_path):
    try:
        return os.path.getsize(video_path)
    except:
        return float('inf')
def select_shortest_video(video_list):
    if not video_list:
        return None
    video_info = []
    for video in video_list:
        duration = get_video_duration(video)
        size = get_file_size(video)
        video_info.append((duration, size, video))
    video_info.sort()
    return video_info[0][2]
def timestamps(video_path, count=1):
    duration = get_video_duration(video_path)
    timestamps = sorted([random.uniform(0, duration) for _ in range(count)])
    return timestamps
#--------------------------------------------------- screenshot -----------------------------------------------------------------------
def screenshot(videos,downloadType,multiple,times,oldfilesremove,separate,outputfolder):
  videos_list = processTheVideos(videos)
  current_output_folder=outputfolder
  if oldfilesremove == 1:
    shutil.rmtree(outputfolder, ignore_errors=True)
  os.makedirs(outputfolder, exist_ok=True)
  time_list=[]
  if times!=None:
    time_list=times.split(",")
    for t in time_list:
        safe_time = t.replace(":", "-")
        onfiledownload=0
        if len(videos_list)>1:
            current_output_folder = os.path.join(outputfolder, safe_time)
            os.makedirs(current_output_folder, exist_ok=True)
        print("folder: ",current_output_folder)
        print("files ------")
        for v in videos_list:
            safe_filename = os.path.splitext(os.path.basename(v))[0]
            filename = os.path.join(current_output_folder, f"{safe_time}_{safe_filename}.png")
            cmd = [
                'ffmpeg','-ss', t,'-i', v,
                '-frames:v', '1',
                '-loglevel', 'error',
                '-hide_banner',
                '-y', filename
            ]
            execution(cmd)
            print(filename)

            if downloadType == 1 and onfiledownload == 0:
                files.download(filename)
                onfiledownload = 1
            elif downloadType == 2:
                files.download(filename)

  remaining=multiple-len(time_list)
  if remaining>0:
    if separate=="1":
      for v in videos_list:
        onfiledownload=0
        if len(videos_list)>1:
            current_output_folder = os.path.join(outputfolder, os.path.splitext(os.path.basename(v))[0])
            os.makedirs(current_output_folder, exist_ok=True)
        print("folder: ",current_output_folder)
        print("files ------")
        times=timestamps(v,remaining)
        for t in times :
          filename = os.path.join(current_output_folder, f"{t}_{os.path.splitext(os.path.basename(v))[0]}.png")
          cmd = ['ffmpeg', '-ss', str(t),'-i', v,'-frames:v', '1', '-loglevel', 'error', '-hide_banner', '-y', filename]
          execution(cmd)
          print(filename)
          if downloadType==1 and onfiledownload==0:
              files.download(filename)
              onfiledownload=1
          elif downloadType==2:
              files.download(filename)
    else:
      os.makedirs(KEYFRAMES_DIR, exist_ok=True)
      shortestvideo = select_shortest_video(videos_list)
      time_list=timestamps(shortestvideo,remaining)
      for t in time_list:
        safe_time = str(t).replace(".", "-")
        onfiledownload=0
        if len(videos_list)>1:
            current_output_folder=os.path.join(outputfolder,safe_time)
            os.makedirs(current_output_folder, exist_ok=True)
        print("folder: ",current_output_folder)
        print("files ------")
        for v in videos_list:
          filename = os.path.join(current_output_folder, f"{safe_time}_{os.path.splitext(os.path.basename(v))[0]}.png")
          cmd = ['ffmpeg', '-ss', str(t), '-i', v, '-frames:v', '1', '-loglevel', 'error', '-hide_banner', '-y', filename]
          execution(cmd)
          print(filename)
          if downloadType==1 and onfiledownload==0:
              files.download(filename)
              onfiledownload=1
          elif downloadType==2:
              files.download(filename)
#@title screenshot
videos = "" #@param {"type":"string","placeholder":"⦿ ex: /videos ⦿ ex: /videos/video1.mk,/videos/video2.mkv ⦿ ex: /videos:video1.mkv,video2.mkv,/pictures:sample1.mkv"}
downloadType = "" #@param {"type":"string","placeholder":"1 means 1st file download 2 means all files download"}
multiple = "" #@param {"type":"string","placeholder":"enter no of screenshots default=1"}
times="" #@param {"type":"string","placeholder":"enter timestamps separated by commas"}
oldfilesremove="" #@param {"type":"string","placeholder":"enter 1 to remove contents in output folder"}
separate="" #@param {"type":"string","placeholder":"enter 1 to take screenshots individually"}
outputfolder="" #@param {"type":"string","placeholder":"enter output folder name or paste path"}
if downloadType=="":
  downloadType=None
if multiple=="":
  multiple=1
if times=="":
  times=None
if oldfilesremove=="":
  oldfilesremove=None
if separate=="":
  separate=None
if outputfolder=="":
  outputfolder="screenshots"
downloadType=int(downloadType)
multiple=int(multiple)
if videos:
  screenshot(videos,downloadType,multiple,times,oldfilesremove,separate,outputfolder)

In [ ]:
#@title save colab files
from google.colab import files
import os

path = ""  #@param {"type":"string","placeholder":"paste file or folder path"}

if os.path.isdir(path):
    for root, dirs, filenames in os.walk(path):
        for filename in filenames:
            filepath = os.path.join(root, filename)
            files.download(filepath)  # each file prompts a download
else:
    files.download(path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#@title Download (standard)
Link = "" # @param {"type":"string","placeholder":"[*] paste link but for gdrive paste id for multiple paste txt file path"}
Type ="" # @param {"type":"string","placeholder":"1.direct link 2.multi direct link 3.google file 4.google folder 5.multiple #drivefile 6.multiple #direct 7.filname multiple 8. mega 9.gofile"}
filename ="" #@param{"type":"string","placeholder":"enter file name for type 1,3 ex: sample.mkv"}
outputfolder = "" #@param {"type":"string","placeholder":"paste output folder path"}
import os
import subprocess
import shutil
import importlib.util
import sys
from datetime import datetime
def is_megatools_installed():
        try:
            # Run 'megadl --version' to check if megatools is available
            subprocess.run(["megadl", "--version"], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
            return True
        except Exception:
            return False
def move_files_to_main(main_folder):
            for root, dirs, files in os.walk(main_folder):
                if root != main_folder:  # Skip the main folder itself
                    for file in files:
                        source_path = os.path.join(root, file)
                        destination_path = os.path.join(main_folder, file)
                        try:
                            shutil.move(source_path, destination_path)
                        except FileExistsError:
                            print(f"Warning: File '{file}' already exists in main folder. Skipping.")
            # Walk again in reverse order and remove empty subdirectories
            for root, dirs, _ in os.walk(main_folder, topdown=False):
                for dir in dirs:
                    subdir_path = os.path.join(root, dir)
                    try:
                        os.rmdir(subdir_path)  # Only removes empty folders
                    except OSError:
                        shutil.rmtree(subdir_path)  # Removes non-empty folders if any
def downloadfunc(Link,Type,outputfolder,filename):
  if outputfolder:
    if not os.path.exists(outputfolder) : os.makedirs(outputfolder)
  Link=Link.strip()
  if Link!="":
    if Type=="1":
        if filename:
              !wget -O "{outputfolder}/{filename}" "{Link}"
        else:
              !wget --content-disposition -P "{outputfolder}" "{Link}"

    elif Type == "2":
      links = Link.split(",")
      for i in links:
          !wget --content-disposition -P "{outputfolder}" "{i}"

    elif Type=="3":
        if filename:
              !gdown -O "{outputfolder}/{filename}" "https://drive.google.com/uc?id={Link}"
        else:
              !gdown -O "{outputfolder}/" "https://drive.google.com/uc?id={Link}"

    elif Type=="4":
        !gdown --folder -O "{outputfolder}/" "https://drive.google.com/drive/folders/{Link}"

    elif Type=="5":
      with open(Link, "r") as f:
        for line in f:
            file_id = line.strip()  # Remove any extra spaces or newlines
            if file_id:  # Ensure it's not an empty line
                !gdown -O "{outputfolder}/" "https://drive.google.com/uc?id={file_id}"

    elif Type == "6":
      with open(Link, "r") as f:
        for line in f:
            url = line.strip()  # Remove any extra spaces or newlines
            !wget --content-disposition -P "{outputfolder}" "{url}"

    elif Type=="7":
      with open(Link, "r") as file:
        lines = [line.strip() for line in file if line.strip()]  # Remove empty lines
      # Go through lines in pairs: (filename, url)
      for i in range(0, len(lines), 2):
          name = lines[i]
          url = lines[i + 1]
          !wget -O "{outputfolder}/{name}" "{url}"

    elif Type == "8":
      if not is_megatools_installed():
          print("megatools not found. Installing...")
          !apt-get update -qq
          !apt-get install -y megatools
      os.chdir(outputfolder)
      if "mega.nz/folder/" in Link.lower():
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        folder_name = f"mega_{timestamp}"
        os.makedirs(folder_name, exist_ok=True)
        os.chdir(folder_name)
      !megadl "{Link}"
      os.chdir("/content")

    elif Type=="9":
        output="python /content/gofile-downloader/gofile-downloader.py "+Link
        if not os.path.exists("/content/gofile-downloader/gofile-downloader.py"):
         os.system('gdown --folder "https://drive.google.com/drive/folders/1WoQjAMndN0iARRj_RYqBwhi9vLV6oDA9"')
        os.chdir(outputfolder)
        os.system(output)
        os.chdir("/content")
        if outputfolder!="/content":
          move_files_to_main(outputfolder)
        os.chdir("/content")

    else:
      print("invalid type")

  else:
    print("no link is provided")
if outputfolder=="":
  outputfolder="temp"
if filename=="":
  filename=None
if Type=="":
  Type="1"
downloadfunc(Link,Type,outputfolder,filename)

In [ ]:
# @title Extract Archives (ZIP, RAR, 7Z, TAR) with Password Support
import os
import subprocess
import shutil

def install_dependencies():
    subprocess.run("apt-get install -y unrar p7zip-full", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def extract_file(file_path, output_folder=None, password=None):
    if not os.path.exists(file_path):
        print(f"❌ Error: File not found: {file_path}")
        return

    # Default output folder based on archive name
    if not output_folder:
        output_folder = os.path.splitext(os.path.basename(file_path))[0]

    os.makedirs(output_folder, exist_ok=True)

    print(f"📦 Extracting: {file_path}")
    file_path_escaped = f'"{file_path}"'
    output_folder_escaped = f'"{output_folder}"'

    if file_path.endswith(".zip"):
        cmd = f'unzip -q '
        if password:
            cmd += f'-P "{password}" '
        cmd += f'{file_path_escaped} -d {output_folder_escaped}'
        result = subprocess.run(cmd, shell=True)

    elif file_path.endswith(".rar"):
        install_dependencies()
        cmd = f'unrar x -o+ '
        if password:
            cmd += f'-p{password} '
        else:
            cmd += '-p- '  # empty password
        cmd += f'{file_path_escaped} {output_folder_escaped}/'
        result = subprocess.run(cmd, shell=True)

    elif file_path.endswith((".tar", ".tar.gz", ".tgz", ".tar.xz", ".tar.bz2")):
        cmd = f'tar -xf {file_path_escaped} -C {output_folder_escaped}'
        result = subprocess.run(cmd, shell=True)

    elif file_path.endswith(".7z"):
        install_dependencies()
        cmd = f'7z x {file_path_escaped} -o{output_folder_escaped}'
        if password:
            cmd += f' -p{password}'
        result = subprocess.run(cmd, shell=True)

    else:
        print("❌ Unsupported file format! Only ZIP, RAR, 7Z, TAR supported.")
        return

    if result.returncode != 0:
        print(f"❌ Extraction failed: {file_path}")
    elif not os.listdir(output_folder):
        print(f"⚠️ Extraction finished but no files found in: {output_folder}")
    else:
        print(f"✅ Success! Extracted to: {output_folder}")

# === Example Usage ===
file_paths = ""  # @param {"type":"string"}
output_folder = ""  # @param {"type":"string"}
password = ""        # @param {"type":"string"}


if os.path.isdir(file_paths):
    file_paths = ",".join([os.path.join(file_paths, f) for f in os.listdir(file_paths) if os.path.isfile(os.path.join(file_paths, f))])
for path in file_paths.split(","):
    extract_file(path.strip(), output_folder, password)

In [ ]:
# @title Folder (standard)
import os
import shutil
import glob
source = "" #@param {"type":"string","placeholder":"[*] enter source path"}
destination = "" #@param {"type":"string","placeholder":"enter destination path"}
operation ="" #@param {"type":"string","placeholder":"[*] values = n:create,c:copy,m:move, s:size,d:delete,df:delete files,cf:copy files ,mf:move files [ for create,delete,size destination not required ]"}
extension="" #@param {"type":"string","placeholder":"which files to select (for df,cf,mf can use) ex: mkv"}
if extension:
  extension = f"*.{extension}" if extension else "*"
else:
  extension="*"
def folderfunc(source,destination,operation):
  if not os.path.exists(destination) and destination:
    os.makedirs(destination)
  for src in source.split(","):
    if operation == "n":
      !mkdir "{src}"
    elif operation == "c":
      !cp -r "{src}" "{destination}"
    elif operation == "m":
      !mv "{src}" "{destination}"
    elif operation == "s":
      !du -sh "{src}"
    elif operation == "d":
      !rm -r "{src}"
    elif operation == "df":
      if os.path.isfile(src):
        os.remove(src)
      else:
        for file in sorted(glob.glob(f"{src}/{extension}")):
          if os.path.isfile(file):
            os.remove(file)
    elif operation == "cf":
      if os.path.isfile(src):
        shutil.copy2(src, destination)
      else:
        for file in sorted(glob.glob(f"{src}/{extension}")):
          if os.path.isfile(file):
            shutil.copy2(file, destination)
    elif operation == "mf":
      if os.path.isfile(src):
        shutil.move(src, destination)
      else:
        for file in sorted(glob.glob(f"{src}/{extension}")):
          if os.path.isfile(file):
            shutil.move(file, destination)
    else:
      print("invalid operation")
folderfunc(source,destination,operation)

In [ ]:
#@title web download
import os, requests, zipfile, subprocess, time, shutil
# --- Install Chrome (only if not installed) ---
if not os.path.exists("/usr/bin/google-chrome"):
    !wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
    !dpkg -i google-chrome-stable_current_amd64.deb || apt-get -f install -y

# --- Detect Chrome version ---
chrome_version = subprocess.getoutput("google-chrome --version").split()[2]
print("Installed Chrome version:", chrome_version)

# --- Get exact matching ChromeDriver version ---
# Use the exact version instead of just major version
driver_version = chrome_version  # Use the same version as Chrome
print("Required ChromeDriver version:", driver_version)

# --- Install ChromeDriver ---
driver_path = "/usr/local/bin/chromedriver"
if not os.path.exists(driver_path) or not subprocess.getoutput(f"{driver_path} --version").startswith(f"ChromeDriver {driver_version}"):
    # Use the official Chrome for Testing download endpoint
    zip_url = f"https://storage.googleapis.com/chrome-for-testing-public/{driver_version}/linux64/chromedriver-linux64.zip"

    try:
        r = requests.get(zip_url)
        r.raise_for_status()

        with open("chromedriver.zip", "wb") as f:
            f.write(r.content)

        with zipfile.ZipFile("chromedriver.zip", "r") as zip_ref:
            zip_ref.extractall("/tmp/")

        # Move to final location
        if os.path.exists("/tmp/chromedriver-linux64/chromedriver"):
            shutil.move("/tmp/chromedriver-linux64/chromedriver", driver_path)
            !chmod +x /usr/local/bin/chromedriver
        else:
            raise Exception("ChromeDriver not found in extracted zip")

    except Exception as e:
        print(f"Failed to download exact version {driver_version}, trying latest compatible version...")
        # Fallback: get the latest version for the major version
        major_version = chrome_version.split(".")[0]
        url = f"https://googlechromelabs.github.io/chrome-for-testing/LATEST_RELEASE_{major_version}"
        driver_version = requests.get(url).text.strip()

        zip_url = f"https://storage.googleapis.com/chrome-for-testing-public/{driver_version}/linux64/chromedriver-linux64.zip"
        r = requests.get(zip_url)

        with open("chromedriver.zip", "wb") as f:
            f.write(r.content)

        with zipfile.ZipFile("chromedriver.zip", "r") as zip_ref:
            zip_ref.extractall("/tmp/")

        shutil.move("/tmp/chromedriver-linux64/chromedriver", driver_path)
        !chmod +x /usr/local/bin/chromedriver

print("ChromeDriver version:", subprocess.getoutput("chromedriver --version"))

# --- Install Selenium (only once) ---
try:
    import selenium
except ImportError:
    !pip install selenium --quiet
    import selenium
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

# --- Setup download directory ---
download_dir = "" #@param{"type":"string","placeholder":"folder name"}
os.makedirs(download_dir, exist_ok=True)

# --- Configure Chrome Options ---
options = Options()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_experimental_option("prefs", {
    "download.default_directory": download_dir,
    "download.prompt_for_download": False,
    "safebrowsing.enabled": True
})

# --- Launch Chrome ---
try:
    driver = webdriver.Chrome(service=Service(driver_path), options=options)
    print("✅ Chrome + Driver started successfully!")
except Exception as e:
    print(f"Error starting Chrome: {e}")
    # Try with different approach
    options.binary_location = "/usr/bin/google-chrome"
    driver = webdriver.Chrome(service=Service(driver_path), options=options)
    print("✅ Chrome + Driver started successfully with explicit binary location!")

# --- Rest of your code remains the same ---
url = "" #@param{"type":"string","placeholder":"paste link"}
driver.get(url)
time.sleep(10)

print("Page title:", driver.title)

# --- Click 'Fast Download Server' ---
TARGET_TEXT = "" #@param{"type":"string"}
elements = driver.find_elements(By.XPATH, f"//*[normalize-space(text())='{TARGET_TEXT}']")
if elements:
    print("✅ Found element with text:", TARGET_TEXT)
    elements[0].click()
else:
    print("❌ No element found with text:", TARGET_TEXT)

# --- Monitor download ---
time.sleep(3)
while True:
    files = [f for f in os.listdir(download_dir) if f.endswith(".crdownload")]
    if not files:
        print("\n✅ Download complete or no active download found.")
        break
    file_path = os.path.join(download_dir, files[0])
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"\r⬇️ Downloading: {files[0]} | Size: {size_mb:.2f} MB", end="", flush=True)
    time.sleep(1)

driver.quit()

In [ ]:
# @title YouTube
import os
url = ""  # @param {"type":"string","placeholder":"[*] paste url link"}
name = ""  # @param {"type":"string","placeholder":"enter name (optional)"}
format = ""  # @param {"type":"string","placeholder":"for select the format, type: yes [default bestvideo+bestaudio]"}
f = ""
audioOnly = ""  # @param {"type":"string","placeholder":"for audio only type: yes"}
folder = ""  # @param {"type":"string","placeholder":"paste output folder path (optional)"}
command = "" # @param {"type":"string","placeholder":"write your own yt-dlp command"}
cookies = ""  # @param {"type":"string","placeholder":"paste cookies path (optional)"}
import os
import subprocess
import sys

try:
    subprocess.run(["yt-dlp", "--version"], check=True)
except FileNotFoundError:
    print("yt-dlp is not installed. Installing now...")
    subprocess.run([sys.executable, "-m", "pip", "install", "yt-dlp"])
if command!="":
  subprocess.run(command, shell=True, check=True)
else :
  if name == "":
    name = "%(title)s.%(ext)s"
  else:
      name += ".%(ext)s"
  if folder != "":
      name = os.path.join(folder, name)

  f=""
  if audioOnly == "yes":
      f = "-x --audio-format mp3 "
  elif format == "yes":
      !yt-dlp --cookies {cookies} -F {url}
      if "input" in locals():
          del input
      f = " -f " + input("enter format ex: 256+303: ")
  if cookies:
    f += f" --cookies \"{cookies}\""
  !yt-dlp -o "{name}" {f} {url}

In [ ]:
# @title compress
import os
import subprocess
import ast
if 'input' in locals():
  del input
# ==== User Parameters ====
input_path = "" # @param {"type":"string","placeholder":"paste input file path"}
output_path = "" # @param {"type":"string","placeholder":"paste output file path"}
input_path = input_path.strip()
output_path = output_path.strip()
codec = ""   # @param {"type":"string","placeholder":"enter codec name options: x264, x265, VP9, AV1"}
if codec=="":
  codec="x265"
preset = "" # @param {"type":"string","placeholder":"enter preset ex: slow"}
if preset=="":
  preset="slow"
filesize=""# @param {"type":"string","placeholder":"enter size in mb ex: 500"}
size_mb=None
if filesize!="":
  size_mb = int(filesize)
crfvalue = "" # @param {"type":"string","placeholder":"Used if filesize is not given default crf value=23"}
crf=23
if crfvalue!="":
  crf=int(crfvalue)
tenbit="" # @param {"type":"string","placeholder":"enter 1 for 10bit"}
use_10bit = False
if tenbit=="1":
  use_10bit = True

resolution = ""  # @param {"type":"string","placeholder":"ex: 1080, 720, 480"}
custom_resolution = ""  # @param {"type":"string","placeholder":"ex: (854, 480)"}
onlyvideo="" # @param {"type":"string","placeholder":"take video only in encoding type yes"}
advance="" # @param {"type":"string","placeholder":"ex: enter extra parameters like filters it will add to command"}
audio_copy = True
subtitles_copy = True

# ==== GPU Detection ====
def has_gpu():
    try:
        subprocess.check_output(['nvidia-smi'], stderr=subprocess.DEVNULL)
        return True
    except Exception:
        return False

gpu_available = has_gpu()

# ==== Bitrate Calculation ====
video_bitrate = None
if size_mb:
    result = subprocess.run(
        ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
         '-of', 'default=noprint_wrappers=1:nokey=1', input_path],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    output = result.stdout.decode().strip()
    for line in output.splitlines()[::-1]:
        try:
            duration = float(line.strip())
            break
        except ValueError:
            continue
    else:
        raise ValueError(f"Could not extract duration. Output:\n{output}")

    size_kb = size_mb * 1024
    video_bitrate = int((size_kb * 8) / duration)  # kbps

# ==== Encoder Selection ====
if codec == 'x264':
    encoder = 'h264_nvenc' if gpu_available else 'libx264'
elif codec == 'x265':
    encoder = 'hevc_nvenc' if gpu_available else 'libx265'
elif codec == 'VP9':
    encoder = 'libvpx-vp9'  # No NVENC support
elif codec == 'AV1':
    encoder = 'libaom-av1'  # No NVENC support
else:
    raise ValueError("Unsupported codec")

# ==== Preset Mapping for NVENC ====
nvenc_preset_map = {
    "ultrafast": "llhp",
    "superfast": "llhp",
    "veryfast": "ll",
    "faster": "fast",
    "fast": "medium",
    "medium": "medium",
    "slow": "slow",
    "slower": "slow",
    "veryslow": "slow",
    "placebo": "slow",
}

actual_preset = nvenc_preset_map.get(preset, preset) if gpu_available and encoder.endswith('_nvenc') else preset

# ==== Resolution Handling ====
scale_filter = None
if custom_resolution:
    width, height = ast.literal_eval(custom_resolution)
    scale_filter = f"scale={width}:{height}"
elif resolution:
    scale_filter = f"scale=-2:{resolution}"  # maintain aspect ratio

# ==== 10-bit Option ====
extra_args = []
if use_10bit:
    if encoder == 'hevc_nvenc':
        extra_args += ['-pix_fmt', 'p010le', '-profile:v', 'main10']
    elif codec == 'x265' and not encoder.endswith('nvenc'):
        extra_args += ['-pix_fmt', 'yuv420p10le']
    elif codec in ['VP9', 'AV1']:
        extra_args += ['-pix_fmt', 'yuv420p10le']
elif codec == 'x264' and encoder == 'h264_nvenc':
    extra_args += ['-pix_fmt', 'yuv420p']


# ==== FFmpeg Command Construction ====
cmd = ['ffmpeg', '-y']
if gpu_available:
    cmd += ['-hwaccel', 'cuda']
cmd += ['-i', input_path]

# Scaling filter
if scale_filter:
    cmd += ['-vf', scale_filter]
if onlyvideo=="yes":
  audio_copy = False
cmd += ['-map', '0:v:0']

# Video encoding
cmd += ['-c:v', encoder, '-preset', actual_preset]

if video_bitrate:
    cmd += ['-b:v', f'{video_bitrate}k']
else:
    cmd += ['-crf', str(crf)]

cmd += extra_args

# Audio and subtitles
if audio_copy == True:
  audio_input = input("Enter audio stream indices (e.g., 0 or 0,1,2 or leave blank for all): ").strip()

  if audio_input == "":
      # Select all audio streams
      cmd += ['-map', '0:a?']
  else:
      # Split and clean indices
      audio_indices = [i.strip() for i in audio_input.split(',')]
      for idx in audio_indices:
          cmd += ['-map', f'0:a:{idx}']

  convert_audio = input("Convert audio? (enter 1): ").strip().lower()
  if convert_audio == "1":
      audio_codec = input("Enter audio codec (e.g., aac, opus): ").strip() or "aac"
      audio_bitrate = input("Enter audio bitrate in kbps (e.g., 256): ").strip() or "256"
      channels = input("Enter number of audio channels (e.g., 2 for stereo, 6 for 5.1) or press Enter to keep original: ").strip()
      cmd += ['-c:a', audio_codec, '-b:a', f"{audio_bitrate}k"]

      if channels:
          cmd += ['-ac', channels]

  else:
      cmd += ['-c:a', 'copy']
# Subtitles
if subtitles_copy:
    cmd += ['-map', '0:s?']
if subtitles_copy:
    cmd += ['-c:s', 'copy']
#advance
if advance:
    cmd += advance.strip().split()
# Output file
cmd += [output_path]

# ==== Run FFmpeg ====
print("Running command:")
print(" ".join(cmd))
result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(result.stdout.decode())

In [ ]:
#@title Multi Functions
#======= imports =======
import re
import subprocess
import os
import json
from natsort import natsorted
import shutil
import sys
#=============================== prerequests ================
if 'input' in locals():
  del input
def install_mkvmerge():
      try:
          # Check if mkvmerge is installed
          subprocess.run(["mkvmerge", "--version"], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
      except FileNotFoundError:
          # If not installed, print a message and install it using apt
          print("mkvmerge is not installed. Installing now...")
          subprocess.run(["sudo", "apt-get", "update", "-qq"], check=True)
          subprocess.run(["sudo", "apt-get", "install", "-y", "mkvtoolnix"], check=True)
      except subprocess.CalledProcessError as e:
          print(f"An error occurred while checking mkvmerge: {e}")
def remove_ipynb_checkpoints(list_of_folders):
  folders=list_of_folders
  for folder in folders:
    if os.path.isdir(folder):
      ipynb_path = os.path.join(folder, ".ipynb_checkpoints")
      if os.path.isdir(ipynb_path):
          shutil.rmtree(ipynb_path)
def install_mediainfo():
  try:
    subprocess.run(["mediainfo", "--version"], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  except FileNotFoundError:
    print("mediainfo is not installed. Installing now...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "mediainfo"], check=True)
#=======================  display streams   ====================================
def parse_mediainfo_fields(filepath):
    result = subprocess.check_output(['mediainfo', filepath], text=True)
    lines = result.splitlines()

    streams = []
    current = {}
    stream_type = ""

    for line in lines:
        if not line.strip():
            continue
        if re.match(r'^(General|Video|Audio|Text)', line):
            if current:
                streams.append((stream_type, current))
            stream_type = line.strip()
            current = {}
        elif ':' in line:
            k, v = map(str.strip, line.split(':', 1))
            current[k] = v
    if current:
        streams.append((stream_type, current))
    return streams
# STEP 4: Clean & Display
def clean_size(size):
    # Remove inbuilt parentheses with percentages
    return re.sub(r'\s*\(.*?\)', '', size).strip()

def clean_resolution(w, h):
    return f"{w.replace(' ', '')}x{h.replace(' ', '')}".replace("pixels", "")

def extract_stream_id(stream_title):
    # extract the last number after '#'
    match = re.search(r'#(\d+)', stream_title)
    return match.group(1) if match else ""

def print_formatted_summary(streams):
    for i,(stream_type, data) in enumerate(streams):
        try:
            id_ = int(str(data.get("ID", ""))) - 1
        except ValueError:
            print() # id_ = i  # fallback index if ID is missing or not numeric
        title = data.get("Title", "")
        size = clean_size(data.get("Stream size", ""))
        duration = data.get("Duration", "")
        bit_depth = data.get("Bit depth", "")
        hdr_tag = data.get("HDR format", "").strip()  # Only use this now
        if stream_type.startswith("Video"):
            res = clean_resolution(data.get("Width", ""), data.get("Height", ""))
            output = f"{id_}: ({data.get('Format', '')}), ({bit_depth}), ({size}), ({res}), ({duration}), ({title})"
            if hdr_tag:  # Only add if HDR format exists
                output += f", ({hdr_tag})"
            print(output)
        elif stream_type.startswith("Audio"):
            lang = data.get("Language", "")
            layout = data.get("Channel layout", "")
            Channel = data.get("Channel(s)", "")
            bitrate = data.get("Bit rate", "")
            fmt = data.get("Format", "")
            profile = data.get("Format profile", "")
            full_fmt = f"{fmt} {profile}".strip()
            stream_name = f"Audio #{extract_stream_id(stream_type)}"
            print(f"{id_}: ({stream_name}) ({lang}), ({size}), ({full_fmt}), ({Channel}), ({bitrate}), ({duration}), ({title}) ,  ({layout})")
        elif stream_type.startswith("Text"):
            lang = data.get("Language", "")
            codec = data.get("Codec ID", "")
            fmt = data.get("Format", "")
            print(f"{id_}: (Subtitle) ({lang}), ({fmt}), ({codec}), ({duration}), ({title})")
def display_streams_func(filepath):
  streams = parse_mediainfo_fields(filepath)
  print("**** streams ******")
  print_formatted_summary(streams)

#======================= execution =========================================
def get_stream_indexes_by_codec(filepath, codec):
    ffprobe_cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "stream=index,codec_name", "-of", "json", filepath
    ]
    data = json.loads(subprocess.check_output(ffprobe_cmd))
    return [str(s["index"]) for s in data["streams"] if s["codec_name"].lower() == codec.lower()]
#================== sub functions =======================================
def create_output_folder(outputfolder):
    # If it's a comma-separated list (merge mode), do NOT try to create a folder.
    if not outputfolder or ("," in outputfolder):
        return outputfolder
    os.makedirs(outputfolder, exist_ok=True)
    return outputfolder

def convert_func(convert):
    cmd = []
    if convert != "":
        convert_c = convert.split(",")
        for c in convert_c:
            if c.startswith("a:"):
                parts = c[2:].split()
                codec = parts[0]
                bitrate = parts[1] if len(parts) > 1 else None
                channels = parts[2] if len(parts) > 2 else None
                if codec!="same":
                  cmd.extend(["-c:a", codec])
                if bitrate:
                    cmd.extend(["-b:a", bitrate])
                    cmd.extend(["-vbr","0"])
                if channels:
                    cmd.extend(["-ac", channels])
            elif c.startswith("s:"):
                codec = c[2:].strip()
                cmd.extend(["-c:s", codec])
    if 'a:' not in convert:
      cmd+=['-c:a','copy']
    if 's:' not in convert:
      cmd+=['-c:s','copy']
    return cmd
def selection_func(audio_selection,subtitle_selection,first_video):
  if audio_selection=="yes" or subtitle_selection=="yes" :
    display_streams_func(first_video)
  cmd = []
  if audio_selection != "":
      if audio_selection == "yes":
          audio_streams = input("Enter audio streams (comma separated): ").split(",")
          for a in audio_streams:
              cmd += ["-map", f"0:{a.strip()}"]
      else:
          audio_s = audio_selection.split(",")
          for a in audio_s:
              if a.startswith("f:"):
                  fmt = a[2:].lower()
                  indexes = get_stream_indexes_by_codec(first_video, fmt)
                  for idx in indexes:
                      cmd += ["-map", f"0:{idx}"]
              elif a.startswith("l:"):
                  lang = a[2:].lower()
                  cmd += ["-map", f"0:a:m:language:{lang}"]
              else:
                  cmd += ["-map", f"0:{a}"]
  else:
    cmd+=['-map','0:a?']
  if subtitle_selection != "":
      if subtitle_selection == "yes":
          subtitle_streams = input("Enter subtitle streams (comma separated): ").split(",")
          for s in subtitle_streams:
              cmd += ["-map", f"0:{s.strip()}"]
      else:
          subtitle_s = subtitle_selection.split(",")
          for s in subtitle_s:
              if s.startswith("f:"):
                  fmt = s[2:].lower()
                  indexes = get_stream_indexes_by_codec(first_video, fmt)
                  for idx in indexes:
                      cmd += ["-map", f"0:{idx}"]
              elif s.startswith("l:"):
                  lang = s[2:].lower()
                  cmd += ["-map", f"0:s:m:language:{lang}"]
              else:
                  cmd += ["-map", f"0:{s}"]
  else:
    cmd+=['-map','0:s?']

  return cmd

def get_audio_stream_indices(video_path):
    try:
        cmd = [
            "ffprobe", "-v", "error",
            "-select_streams", "a",
            "-show_entries", "stream=index",
            "-of", "json", video_path
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        data = json.loads(result.stdout)

        if "streams" not in data or not data["streams"]:
            return ""

        indices = [str(stream["index"]) for stream in data["streams"]]
        return ",".join(indices)

    except subprocess.CalledProcessError as e:
        print("Error:", e.stderr)
        return ""

#======================= functions ==========================================
def one_input(video,audio,subtitle,title,filename,outputfolder,input1,audio_delay,audio_selection,convert,subtitle_selection,startt):
  if os.path.isdir(input1):
    video_files =  natsorted([ os.path.join(input1, f) for f in os.listdir(input1) if os.path.isfile(os.path.join(input1, f))])
  else:
    video_files = [input1]
  first_video = video_files[0]
  i=0
  selection=selection_func(audio_selection,subtitle_selection,first_video)
  conversion=convert_func(convert)
  output = [None] * len(video_files)
  if startt=="":
    startt="1"
  for number, video_file in enumerate(video_files, start=int(startt)):
      formatted_number = f"{number:02d}"
      current_title = title.format(number=formatted_number)
      current_filename = filename.format(number=formatted_number)
      input_video_path = video_file
      output_video_path = os.path.join(outputfolder, current_filename)
      cmd = ['ffmpeg', '-i', input_video_path, '-map', '0:v']
      cmd+=selection
      cmd+=['-c:v','copy']
      cmd+=conversion
      if video != "":
          cmd.extend(["-metadata:s:v", f"title={video}"])
      if audio != "":
          cmd.extend(["-metadata:s:a", f"title={audio}"])
      if subtitle != "":
          cmd.extend(["-metadata:s:s", f"title={subtitle}"])
      if current_title != "":
          cmd.extend(["-metadata", f"title={current_title}"])
      #output
      current_filename=os.path.join(outputfolder,current_filename)
      temp=current_filename
      if convert!="" or audio_delay!="":
        temp="f"+os.path.basename(video_file)
      cmd += ['-y', temp]
      result = subprocess.run(cmd, capture_output=True, text=True)
      if result.returncode != 0:
          print("FFmpeg error:", result.stdout,result.stderr)
      else:
        if convert!="" or audio_delay!="":
          if audio_delay!="":
            if audio_selection=="":
              display_streams_func(temp)
              ad_streams = input("Enter audio delay streams (IDs) separated by commas: ").split(",")
            else:
              ad_streams=get_audio_stream_indices(temp)
          cmd=['mkvmerge','-o',current_filename]
          if audio_delay != "":
            # mkvmerge expects milliseconds (integers), so round it
            audio_delay = int(round(float(audio_delay)))
            for tid in ad_streams:
                tid = tid.strip()
                if tid.isdigit():
                    cmd += ["--sync", f"{tid}:{audio_delay}"]
          cmd.append(temp)
          result=subprocess.run(cmd,capture_output=True, text=True)
          if result.returncode != 0:
            print("mkvmerge error:", result.stdout,result.stderr)
          else:
            output[i]=current_filename
          os.remove(temp)
        else:
          output[i]=current_filename
        i=i+1
  return output

def mergefile(video, audio, subtitle, title, filename, outputfolder, audio_selection, convert, subtitle_selection):
    # Decide input list
    if outputfolder and ("," in outputfolder or not os.path.isdir(outputfolder)):
        # Treat as explicit list
        video_files = [p.strip() for p in outputfolder.split(",") if p.strip()]
        if not video_files:
            raise ValueError("No input files provided for merge.")
        # Put output next to the first input by default
        out_dir = os.path.dirname(video_files[0]) or "."
        merged_name = filename.replace("E{number}", "")
        mergefile_path = os.path.join(out_dir, merged_name)
    else:
        # Treat as a directory containing the parts
        video_files = natsorted([
            os.path.join(outputfolder, f)
            for f in os.listdir(outputfolder)
            if os.path.isfile(os.path.join(outputfolder, f))
        ])
        if not video_files:
            raise ValueError(f"No files found to merge in folder: {outputfolder!r}")
        mergefile_path = os.path.join(outputfolder, filename.replace("E{number}", ""))

    print(video_files)  # keep your debug

    # Write concat list with absolute paths
    merge_list_path = "merge_list.txt"
    with open(merge_list_path, "w") as f:
        for vf in video_files:
            f.write(f"file '{os.path.abspath(vf)}'\n")

    # Build ffmpeg command
    cmd = ['ffmpeg', '-f', 'concat', '-safe', '0', '-i', merge_list_path, '-map', '0:v']

    # For stream selection, inspect the first real file
    selection = selection_func(audio_selection, subtitle_selection, video_files[0])
    cmd += selection
    cmd += ['-c:v', 'copy']
    cmd += convert_func(convert)

    if video:
        cmd += ["-metadata:s:v", f"title={video}"]
    if audio:
        cmd += ["-metadata:s:a", f"title={audio}"]
    if subtitle:
        cmd += ["-metadata:s:s", f"title={subtitle}"]
    if title:
        cmd += ["-metadata", f"title={title.replace('E{number}', '')}"]

    # Output handling
    temp_out = mergefile_path if convert == "" else os.path.join(os.path.dirname(mergefile_path), "f" + os.path.basename(mergefile_path))
    cmd += ['-y', temp_out]

    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("FFmpeg error:", result.stdout, result.stderr)
        return None

    if convert != "":
        result = subprocess.run(['mkvmerge', '-o', mergefile_path, temp_out], capture_output=True, text=True)
        if result.returncode != 0:
            print("mkvmerge error:", result.stdout, result.stderr)
            return None
        os.remove(temp_out)

    return mergefile_path
def two_input(video,audio,subtitle,title,filename,outputfolder,input1,audio_delay,input2,audio_selection,convert,subtitle_selection,startt):
  tempfolder="tempfolder"
  shutil.rmtree(tempfolder, ignore_errors=True)
  os.makedirs(tempfolder, exist_ok=True)
  if os.path.isdir(input1):
    video_files = natsorted([ os.path.join(input1, f) for f in os.listdir(input1) if os.path.isfile(os.path.join(input1, f))])
  else:
    video_files = [input1]
  if os.path.isdir(input2):
    audio_files =natsorted([ os.path.join(input2, f) for f in os.listdir(input2) if os.path.isfile(os.path.join(input2, f))])
  else:
    audio_files = [input2]
  if startt=="": startt="1"
  for number, (video_file, audio_file) in enumerate(zip(video_files, audio_files), start=int(startt)):
    output_file=f"{number}.mkv"
    output_file=os.path.join(tempfolder,output_file)
    cmd=['mkvmerge','-o',output_file,video_file,'--no-video',audio_file]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
      print("mkvmerge error:", result.stdout,result.stderr)
      return None
  output=one_input(video,audio,subtitle,title,filename,outputfolder,tempfolder,audio_delay,audio_selection,convert,subtitle_selection,startt)
  shutil.rmtree(tempfolder)
  return output
#=========================== parameters ==================================
audio_selection = "" #@param{"type":"string","placeholder":"f:eac3,f:aac,1,l:tel,yes"}
subtitle_selection = "" #@param{"type":"string","placeholder":"f:mov,0,l:eng,yes"}
convert = "" #@param{"type":"string","placeholder":"a:libopus 64k 2,s:mov"}

video = ""#@param{"type":"string","placeholder":"enter name for video track"}
audio = "" #@param{"type":"string","placeholder":"enter name for audio tracks"}
subtitle = "" #@param{"type":"string","placeholder":"enter name for subtitle tracks"}
title = "" #@param{"type":"string","placeholder":"enter title"}
filename = "" #@param{"type":"string","placeholder":"enter filename"}
outputfolder = "" #@param{"type":"string","placeholder":"Paste folder path or enter new folder name to save:"}
first_input = "" #@param{"type":"string","placeholder":"paste file1 or folder1 path"}
second_input = "" #@param{"type":"string","placeholder":"paste file2 or folder2 path"}
Audio_delay = "" #@param{"type":"string","placeholder":"enter in milliseconds ex: 1000 = 1s"}
functions="" #@param{"type":"string","placeholder":"[ functions: 1_input,2_input,merge ] enter functions separated by commas"}
start="" #@param{"type":"string","placeholder":"enter starting number for {number} default=1"}

video=video.strip()
audio=audio.strip()
subtitle=subtitle.strip()
title=title.strip()
filename=filename.strip()
outputfolder=outputfolder.strip()
first_input=first_input.strip()
second_input=second_input.strip()
Audio_delay=Audio_delay.strip()
functions=functions.strip()
start=start.strip()

selected_functions=functions.split(",")
outputfolder=create_output_folder(outputfolder)
remove_ipynb_checkpoints([outputfolder,first_input,second_input])
install_mkvmerge()
install_mediainfo()
if "1_input" in selected_functions:
    output=one_input(video,audio,subtitle,title,filename,outputfolder,first_input,Audio_delay,audio_selection,convert,subtitle_selection,start)
    for i in output:
      print(i)
if "2_input" in selected_functions:
    output=two_input(video,audio,subtitle,title,filename,outputfolder,first_input,Audio_delay,second_input,audio_selection,convert,subtitle_selection,start)
    for i in output:
      print(i)
if "merge" in selected_functions:
    if ("1_input" in selected_functions or "2_input" in selected_functions):
      selection=""
      convert_audio=""
    output=mergefile(video,audio,subtitle,title,filename,outputfolder,audio_selection,convert,subtitle_selection)
    print(output)